# Проект модуля. Нейросеть для предсказания калорийности блюд

In [115]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Метаданные

## флаги отладки

## глобальные параметры

In [116]:
# файловая структура проекта
dataset_dir = os.path.join('.','data','project')

# Этап 1. EDA исходного датасета 

## `images` - папки с фото блюд

In [117]:
dish_list = os.listdir(os.path.join(dataset_dir,'images'))

len(dish_list), dish_list[:5]

(3490,
 ['dish_1556572657',
  'dish_1556573514',
  'dish_1556575014',
  'dish_1556575083',
  'dish_1556575124'])

## `dish.csv` - список блюд

In [118]:
dish_df = pd.read_csv(os.path.join(dataset_dir,'dish.csv')).sort_values(['split','dish_id']).reset_index(drop=True)

dish_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3262 entries, 0 to 3261
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   dish_id         3262 non-null   object 
 1   total_calories  3262 non-null   float64
 2   total_mass      3262 non-null   float64
 3   ingredients     3262 non-null   object 
 4   split           3262 non-null   object 
dtypes: float64(2), object(3)
memory usage: 127.5+ KB


In [119]:
dish_df.head()

,dish_id,total_calories,total_mass,ingredients,split
0,dish_1556575327,103.300003,152.0,ingr_0000000034;ingr_0000000012;ingr_0000000016,test
1,dish_1557861216,0.000000,1.0,ingr_0000000423,test
2,dish_1557862345,103.949997,63.0,ingr_0000000028,test
3,dish_1557862696,19.250000,77.0,ingr_0000000007,test
4,dish_1557862738,319.779999,118.0,ingr_0000000010,test


### а все ли блюда есть в списках и на фото?

In [120]:
counter = 0

for i, dish in enumerate(dish_list):
    if dish not in list(dish_df.dish_id):
        counter += 1

if counter == 0:
    print('FYI: All dishes in dish dir list are present in dish.csv(field "dish_id")')
else:
    print('FYI:', counter, 'out of', len(dish_list), ' dishes in dish dir list are missing in dish.csv(field "dish_id")')

FYI: 228 out of 3490  dishes in dish dir list are missing in dish.csv(field "dish_id")


In [121]:
len(dish_df) + counter,  len(dish_list)     # проверяем возможные причины - должны совпасть!

(3490, 3490)

In [122]:
counter = 0

for i, dish in enumerate(list(dish_df.dish_id)):
    if dish not in dish_list:
        counter += 1

if counter == 0:
    print('FYI: All dishes in dish.csv(field "dish_id") are present in dish dir list')
else:
    print('FYI:', counter, 'out of', len(dish_df), ' dishes in dish.csv(field "dish_id") are missing in dish dir list')

FYI: All dishes in dish.csv(field "dish_id") are present in dish dir list


**ВЫВОДЫ:**
* 228 папок с фото лишние в датасете - для них нет информации в списке блюд
* для всех блюд в списке есть папка с фото (поэтому лишние фото не помешают строить детесет, исходя из списка блюд)

## ingredients.csv

In [123]:
ingr_df = pd.read_csv(os.path.join(dataset_dir,'ingredients.csv'))

ingr_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      555 non-null    int64 
 1   ingr    555 non-null    object
dtypes: int64(1), object(1)
memory usage: 8.8+ KB


In [124]:
ingr_df.head()

,id,ingr
0,1,cottage cheese
1,2,strawberries
2,3,garden salad
3,4,bacon
4,5,potatoes


### ID ингридиентов имеют разный формат, проверим гипотезу:
если отбросить все знаки левее первой отличной от нуля цыфры, то получим целый ID из `indredients.csv` колонка `id`

In [125]:
dish_df['dish_ingr_list'] = dish_df.ingredients.map(lambda s: s.strip().split(';'))

In [126]:
dish_df.dish_ingr_list = dish_df.dish_ingr_list.map(pd.Series).map(
                            lambda ser: ser.map(
                                lambda s: int(re.sub(r"^0+", "", re.sub(r"\D", "", s)))
                            )
                        ).map(list)

In [127]:
dish_df.head()

,dish_id,total_calories,total_mass,ingredients,split,dish_ingr_list
0,dish_1556575327,103.300003,152.0,ingr_0000000034;ingr_0000000012;ingr_0000000016,test,"[34, 12, 16]"
1,dish_1557861216,0.000000,1.0,ingr_0000000423,test,[423]
2,dish_1557862345,103.949997,63.0,ingr_0000000028,test,[28]
3,dish_1557862696,19.250000,77.0,ingr_0000000007,test,[7]
4,dish_1557862738,319.779999,118.0,ingr_0000000010,test,[10]


In [128]:
ingr_list = list(set(dish_df.dish_ingr_list.sum()))

len(ingr_list)

200